In [22]:
import os
import platform
import shutil
import psutil
import subprocess
import torch
import json
import peft
import trl
import transformers
import scipy
import accelerate
import datasets
import numpy as np

In [23]:
def test_compute_gpu():
    x = torch.rand(10000, 2).cuda()
    A = torch.rand(10000,2,2).cuda()
    y = torch.bmm(A, x.unsqueeze(2)).squeeze(2)
    torch.cuda.synchronize() 
    return y

In [27]:
computer_state={}
if os.name=='nt':
    computer_state['os']='Windows'
    print(computer_state['os'])
else:
    computer_state['os']=platform.system()
    print(computer_state['os'])

computer_state['python']=platform.python_version()
print("Python version:", computer_state['python'])

if shutil.which('nvidia-smi'):
    print('nvidia-smi disponible')
    computer_state['nvidia-smi']=True
    if(torch.cuda.is_available()):
        computer_state['cuda']=True
        print(torch.cuda.get_device_name(0))
        computer_state['gpu_name']=torch.cuda.get_device_name(0)
        try:
            test_compute_gpu()
            computer_state['gpu_compute_test']=True
            print("GPU computation test passed.")
        except Exception as e:
            print(f"Error occurred while testing GPU computation: {e}")
            computer_state['gpu_compute_test']=False

        computer_state['pytorch_version']=torch.__version__
        computer_state['cuda_version']=torch.version.cuda
        print("PyTorch Version:", computer_state['pytorch_version'])
        print("CUDA Version:", computer_state['cuda_version'])

        result = subprocess.check_output(['nvidia-smi',"--query-gpu=name,memory.total,memory.free,driver_version",
                                "--format=csv,noheader"])
        print("GPU Information:")
        print("name, memory total, memory free, driver version")
        print(result.decode('utf-8'))

        computer_state['gpu_info']={}
        computer_state['gpu_info']['name']=result.decode('utf-8').split(',')[0]
        computer_state['gpu_info']['memory_total']=result.decode('utf-8').split(',')[1]
        computer_state['gpu_info']['memory_free']=result.decode('utf-8').split(',')[2]
        computer_state['gpu_info']['driver_version']=result.decode('utf-8').split(',')[3]
    else:
        print("CUDA for PyTorch is not available. Please check your GPU and CUDA installation.")

        result = subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total,memory.free,driver_version', '--format=csv,noheader'])
        print("GPU Information:")
        print("name, memory total, memory free, driver version")
        print(result.decode('utf-8'))   
        computer_state['gpu_info']={}
        computer_state['gpu_info']['name']=result.decode('utf-8').split(',')[0]
        computer_state['gpu_info']['memory_total']=result.decode('utf-8').split(',')[1]
        computer_state['gpu_info']['memory_free']=result.decode('utf-8').split(',')[2]
        computer_state['gpu_info']['driver_version']=result.decode('utf-8').split(',')[3]

    computer_state['transformers_version']=transformers.__version__
    computer_state['peft_version']=peft.__version__ 
    computer_state['trl_version']=trl.__version__
    computer_state['scipy_version']=scipy.__version__
    computer_state['numpy_version']=np.__version__
    computer_state['accelerate_version']=accelerate.__version__
    computer_state['datasets_version']=datasets.__version__

    print("Transformers Version:", computer_state['transformers_version'])
    print("PEFT Version:", computer_state['peft_version'])
    print("TRL Version:", computer_state['trl_version'])
    print("Scipy Version:", computer_state['scipy_version'])
    print("NumPy Version:", computer_state['numpy_version'])
    print("Accelerate Version:", computer_state['accelerate_version'])
    print("Datasets Version:", computer_state['datasets_version'])

    print('')
    mem = psutil.virtual_memory()
    print("Memory Information:")
    print(f"Total: {mem.total/1024**3:.2f} GB")
    print(f"Available: {mem.available/1024**3:.2f} GB")
    print(f"Used: {mem.used/1024**3:.2f} GB")
    print(f"Percentage: {mem.percent}%")

    computer_state['memory_info']={}
    computer_state['memory_info']['total']=mem.total
    computer_state['memory_info']['available']=mem.available
    computer_state['memory_info']['used']=mem.used
    computer_state['memory_info']['percent']=mem.percent

    print("CUDA Memory Information by Torch:")
    a, b = torch.cuda.mem_get_info()
    print(f"Total: {b/1024**3:.2f} GB")
    print(f"Available: {a/1024**3:.2f} GB")
    print(f"Used: {(b-a)/1024**3:.2f} GB")

    computer_state['cuda_memory_info']={}
    computer_state['cuda_memory_info']['total']=b
    computer_state['cuda_memory_info']['available']=a
    computer_state['cuda_memory_info']['used']=b-a

else:
    computer_state['nvidia-smi']=False
    print('No nvidia-smi detected')

os.makedirs('runs', exist_ok=True)
json_file_path = os.path.join(os.getcwd(),'runs', 'computer_state.json')
with open(json_file_path, 'w') as json_file:
    json.dump(computer_state, json_file, indent=4)

Windows
Python version: 3.11.4
nvidia-smi disponible
NVIDIA GeForce RTX 4060
GPU computation test passed.
PyTorch Version: 2.11.0+cu128
CUDA Version: 12.8
GPU Information:
name, memory total, memory free, driver version
NVIDIA GeForce RTX 4060, 8188 MiB, 5657 MiB, 610.88

Transformers Version: 5.22.1
PEFT Version: 0.21.2
TRL Version: 1.17.1
Scipy Version: 1.11.3
NumPy Version: 1.26.4
Accelerate Version: 1.15.0
Datasets Version: 4.8.5

Memory Information:
Total: 31.92 GB
Available: 11.19 GB
Used: 20.73 GB
Percentage: 64.9%
CUDA Memory Information by Torch:
Total: 8.00 GB
Available: 6.91 GB
Used: 1.09 GB
